# Regional analysis with pycensuskr

In [ ]:
from pycensuskr import CensusKR
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

census = CensusKR()


## Example 1: Population Change (2010 to 2020)


In [ ]:
import matplotlib.pyplot as plt

adm2_2020 = census.load_districts(2020)
df_2010_pop = census.anycensus(year=2010, codes=['Gyeongsangnam-do', 'Busan', 'Ulsan'], type='population')
df_2020_pop = census.anycensus(year=2020, codes=['Gyeongsangnam-do', 'Busan', 'Ulsan'], type='population')

sf_target = (
    adm2_2020
    .merge(df_2010_pop[['adm2_code', 'all households_total_prs']], on='adm2_code', how='inner')
    .merge(df_2020_pop[['adm2_code', 'all households_total_prs']], on='adm2_code', how='inner', suffixes=('_2010', '_2020'))
)
sf_target['change'] = sf_target['all households_total_prs_2020'] - sf_target['all households_total_prs_2010']

ax = sf_target.plot(column='change', cmap='RdBu_r', legend=True, linewidth=0.2, edgecolor='gray', figsize=(8, 6))
ax.set_title('Population change (2010-2020)')
ax.set_axis_off()
plt.show()


## Example 2: Population vs Tax (Bivariate approximation)


In [ ]:
df_2020_pop = census.anycensus(year=2020, type='population')
df_2020_tax = census.anycensus(year=2020, type='tax')

sf_pop = adm2_2020.merge(df_2020_pop[['adm2_code', 'all households_total_prs']], on='adm2_code', how='left')
sf_final = sf_pop.merge(df_2020_tax[['adm2_code', 'income_general_mkr']], on='adm2_code', how='left')

# Quantile classes (3x3 concept)
sf_final['pop_q'] = pd.qcut(sf_final['all households_total_prs'], 3, labels=[1, 2, 3], duplicates='drop')
sf_final['tax_q'] = pd.qcut(sf_final['income_general_mkr'], 3, labels=[1, 2, 3], duplicates='drop')
sf_final['bi_class'] = sf_final['pop_q'].astype(str) + '-' + sf_final['tax_q'].astype(str)

ax = sf_final.plot(column='bi_class', categorical=True, legend=True, figsize=(8, 6), linewidth=0.1, edgecolor='none')
ax.set_title('Population vs Tax (3x3 quantile classes)')
ax.set_axis_off()
plt.show()


## Example 3: Sex Ratio in the Seoul Metropolitan Area


In [ ]:
df_sma = census.anycensus(year=2020, codes=['Seoul', 'Gyeonggi-do', 'Incheon'], type='population')
df_sma['sex_ratio'] = (df_sma['all households_male_prs'] / df_sma['all households_female_prs']) * 100

df_sma[['sex_ratio']].hist(bins=20, figsize=(6, 3))
plt.title('Sex ratio distribution (SMA, 2020)')
plt.xlabel('Sex ratio (male per 100 females)')
plt.show()

sma_map = adm2_2020.merge(df_sma[['adm2_code', 'sex_ratio']], on='adm2_code', how='inner')
ax = sma_map.plot(column='sex_ratio', cmap='RdBu_r', legend=True, linewidth=0.1, edgecolor='gray', figsize=(8, 5))
ax.set_title('Sex ratio in the Seoul Metropolitan Area (2020)')
ax.set_axis_off()
plt.show()


## Example 4: Socioeconomic Profiling with PCA


In [ ]:
from functools import reduce
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load multiple indicators
hou = census.anycensus(year=2020, type='housing', level='adm2')
pop = census.anycensus(year=2020, type='population', level='adm2')
mort = census.anycensus(year=2020, type='mortality', level='adm2')
ss = census.anycensus(year=2020, type='social security', level='adm2')

keys = ['adm1', 'adm1_code', 'adm2', 'adm2_code', 'year']
df_wide = reduce(lambda x, y: x.merge(y, on=keys, how='left'), [hou, pop, mort, ss])

# Derive compact indicators (column names may vary by data release)
ind = pd.DataFrame({
    'adm2_code': df_wide['adm2_code'],
    'adm2': df_wide['adm2'],
})
if 'all households_total_prs' in df_wide and 'housing types_total_cnt' in df_wide:
    ind['persons_per_housing'] = df_wide['all households_total_prs'] / df_wide['housing types_total_cnt']
if 'all households_male_prs' in df_wide and 'all households_female_prs' in df_wide:
    ind['sex_ratio'] = 100 * df_wide['all households_male_prs'] / df_wide['all households_female_prs']
if 'all causes_total_p1p' in df_wide:
    ind['mortality_rate'] = df_wide['all causes_total_p1p']
if 'fertility_total_brt' in df_wide:
    ind['fertility_rate'] = df_wide['fertility_total_brt']

num = ind.select_dtypes('number').drop(columns=['adm2_code'], errors='ignore').dropna()

X = StandardScaler().fit_transform(num)
pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(pcs[:, 0], pcs[:, 1], s=10, alpha=0.7)
plt.title('PCA of socioeconomic indicators')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print('Explained variance ratio:', pca.explained_variance_ratio_)
